# MNIST classification via SVD

This notebook explores a simple linear-algebra classifier for MNIST digits. The main workflow is:

1. Download and load the MNIST dataset.
2. Flatten each 28 by 28 image into a 784-dimensional row vector.
3. Group the training data by digit and compute an SVD for each digit-specific matrix.
4. Use the leading right singular vectors as a low-dimensional model for each digit class.
5. Classify test images by reconstruction error.
6. Inspect a few visual summaries of the resulting subspaces.


In [ ]:
from __future__ import annotations

import gzip
import struct
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw


In [ ]:
MNIST_FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}

MNIST_BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
DATA_DIR = Path("data/mnist")
PREVIEW_PATH = DATA_DIR / "mnist_preview.png"


## Data-loading helpers

These helpers download the compressed MNIST files if needed, parse the IDX file format, and generate a quick preview image.


In [ ]:
def download_file(filename: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        print(f"Using cached file: {destination}")
        return

    url = MNIST_BASE_URL + filename
    print(f"Downloading {url} -> {destination}")
    urllib.request.urlretrieve(url, destination)


def parse_images(path: Path) -> np.ndarray:
    with gzip.open(path, "rb") as handle:
        magic, count, rows, cols = struct.unpack(">IIII", handle.read(16))
        if magic != 2051:
            raise ValueError(f"Unexpected image magic number in {path}: {magic}")
        data = np.frombuffer(handle.read(), dtype=np.uint8)
    return data.reshape(count, rows, cols)


def parse_labels(path: Path) -> np.ndarray:
    with gzip.open(path, "rb") as handle:
        magic, count = struct.unpack(">II", handle.read(8))
        if magic != 2049:
            raise ValueError(f"Unexpected label magic number in {path}: {magic}")
        data = np.frombuffer(handle.read(), dtype=np.uint8)
    if len(data) != count:
        raise ValueError(f"Label count mismatch in {path}: expected {count}, got {len(data)}")
    return data


def load_mnist(data_dir: Path = DATA_DIR) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    paths = {key: data_dir / filename for key, filename in MNIST_FILES.items()}

    for filename in MNIST_FILES.values():
        download_file(filename, data_dir / filename)

    x_train = parse_images(paths["train_images"])
    y_train = parse_labels(paths["train_labels"])
    x_test = parse_images(paths["test_images"])
    y_test = parse_labels(paths["test_labels"])
    return x_train, y_train, x_test, y_test


def create_preview_grid(
    images: np.ndarray,
    labels: np.ndarray,
    destination: Path,
    count: int = 12,
    columns: int = 4,
    scale: int = 6,
    padding: int = 18,
) -> None:
    count = min(count, len(images))
    rows = (count + columns - 1) // columns
    digit_size = images.shape[1] * scale
    canvas_width = columns * digit_size
    canvas_height = rows * (digit_size + padding)

    canvas = Image.new("L", (canvas_width, canvas_height), color=255)
    draw = ImageDraw.Draw(canvas)

    for idx in range(count):
        row, col = divmod(idx, columns)
        x0 = col * digit_size
        y0 = row * (digit_size + padding)
        digit = Image.fromarray(images[idx], mode="L").resize(
            (digit_size, digit_size), resample=Image.Resampling.NEAREST
        )
        canvas.paste(digit, (x0, y0))
        draw.text((x0, y0 + digit_size + 2), f"label={int(labels[idx])}", fill=0)

    destination.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(destination)
    print(f"Saved preview image: {destination}")


## Plotting and classifier helpers

These are the reusable helpers used later in the notebook for visualizing principal components and testing classifiers built from the top `m` PCs.


In [ ]:
def show_pc_grid(Vh: dict[int, np.ndarray], digit: int, rows: int = 2, cols: int = 3, cmap: str = "seismic") -> None:
    pcs = Vh[digit][: rows * cols]
    m = np.max(np.abs(pcs))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

    for j, ax in enumerate(axes.flat):
        ax.imshow(pcs[j].reshape(28, 28), cmap=cmap, vmin=-m, vmax=m)
        ax.set_title(f"PC {j + 1}")
        ax.axis("off")

    fig.suptitle(f"Top {rows * cols} principal components for digit {digit}")
    plt.tight_layout()
    plt.show()


def predict_with_m_pcs(x_test_flat: np.ndarray, Vh: dict[int, np.ndarray], m: int) -> tuple[np.ndarray, np.ndarray]:
    Tm = {d: Vh[d][:m] for d in range(10)}
    x_norm_sq = np.sum(x_test_flat**2, axis=1)
    errors = np.empty((len(x_test_flat), 10))

    for k in range(10):
        Tk = Tm[k]
        coeffs = x_test_flat @ Tk.T
        errors[:, k] = x_norm_sq - np.sum(coeffs**2, axis=1)

    y_pred = np.argmin(errors, axis=1)
    return errors, y_pred


## Load the dataset


In [ ]:
x_train, y_train, x_test, y_test = load_mnist()

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test: ", x_test.shape)
print("y_test: ", y_test.shape)


In [ ]:
create_preview_grid(x_train, y_train, PREVIEW_PATH)
Image.open(PREVIEW_PATH)


## Reshape and organize the images

Each MNIST image is converted from a 28 by 28 grid into a row vector of length 784.


In [ ]:
x_train_flat = x_train.reshape(x_train.shape[0], 784).astype(np.float64)
x_test_flat = x_test.reshape(x_test.shape[0], 784).astype(np.float64)

digits_flat = {d: x_train_flat[y_train == d] for d in range(10)}

print("Flattened training matrix:", x_train_flat.shape)
print("Flattened test matrix:    ", x_test_flat.shape)
print("Digit-0 matrix:           ", digits_flat[0].shape)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(6, 4))

for ax, img in zip(axes.ravel(), digits_flat[0][:6]):
    ax.imshow(img.reshape(28, 28), cmap="gray")
    ax.axis("off")

fig.suptitle("First 6 training images of digit 0")
plt.tight_layout()
plt.show()


## Compute a digit-wise SVD

For each digit `d`, the matrix `digits_flat[d]` contains one flattened training image per row. The right singular vectors of this matrix provide principal directions in the 784-dimensional pixel space.


In [ ]:
svds = {d: np.linalg.svd(digits_flat[d], full_matrices=False) for d in range(10)}
Vh = {d: svds[d][2] for d in range(10)}
S = {d: svds[d][1] for d in range(10)}


In [ ]:
show_pc_grid(Vh, 0, rows=1, cols=2)


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(12, 6))

for j, ax in enumerate(axes.flat):
    digit = j // 2 + 1
    pc = j % 2
    vec = Vh[digit][pc].reshape(28, 28)
    m = np.max(np.abs(vec))
    ax.imshow(vec, cmap="seismic", vmin=-m, vmax=m)
    ax.set_title(f"PC {pc + 1} of digit {digit}")
    ax.axis("off")

fig.suptitle("First 2 principal components of digits 1 through 9")
plt.tight_layout()
plt.show()


Below I demonstrate how linear combinations of the first two principal components morph the digit-0 image.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

pc1 = Vh[0][0].reshape(28, 28)
pc2 = Vh[0][1].reshape(28, 28)

for j, ax in enumerate(axes.flat):
    alpha = -0.4 + 0.1 * j
    vec = (1 - alpha) * pc1 + alpha * pc2
    m = np.max(np.abs(vec))

    ax.imshow(vec, cmap="seismic", vmin=-m, vmax=m)
    ax.set_title(rf"$\alpha = {alpha:.2f}$")
    ax.axis("off")

fig.suptitle(r"$(1-\alpha)PC_1 + \alpha PC_2$ for digit 0")
plt.tight_layout()
plt.show()


## A simple subspace classifier

For each digit `k`, we compare how well the subspace spanned by the first few right singular vectors reconstructs a test image `x`.

$$
\hat d(x) = \arg\min_k \|x - T_k^T T_k x\|^2
$$


In [ ]:
errors6, y_pred6 = predict_with_m_pcs(x_test_flat, Vh, 6)
accuracy6 = np.mean(y_pred6 == y_test)
print(f"Accuracy with 6 PCs per digit: {accuracy6:.4f}")

errors20, y_pred20 = predict_with_m_pcs(x_test_flat, Vh, 20)
accuracy20 = np.mean(y_pred20 == y_test)
print(f"Accuracy with 20 PCs per digit: {accuracy20:.4f}")


In [ ]:
ms = np.arange(1, 51)
accuracies = []

for m in ms:
    _, y_pred = predict_with_m_pcs(x_test_flat, Vh, m)
    accuracies.append(np.mean(y_pred == y_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ms, accuracies, marker="o", linewidth=2)
ax.set_xlabel("Number of principal components per digit")
ax.set_ylabel("Classification accuracy")
ax.set_title("MNIST accuracy vs. number of PCs")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
confusion = np.zeros((10, 10), dtype=int)
for truth, pred in zip(y_test, y_pred20):
    confusion[truth, pred] += 1

confusion_pct = 100 * confusion / confusion.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(confusion_pct, cmap="viridis")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("Predicted digit")
ax.set_ylabel("True digit")
ax.set_title("Confusion matrix for the 20-PC classifier")

for i in range(10):
    for j in range(10):
        color = "white" if confusion_pct[i, j] > 50 else "black"
        ax.text(j, i, f"{confusion_pct[i, j]:.1f}", ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## Inspect a single test image and its subspace errors

Here I inspect one test image and compare the ten reconstruction errors produced by the 20-PC classifier.


In [ ]:
example_idx = 0
x = x_test_flat[example_idx]

plt.imshow(x.reshape(28, 28), cmap="gray")
plt.title(f"Test image {example_idx} (true digit {y_test[example_idx]})")
plt.axis("off")
plt.show()


In [ ]:
example_errors = errors20[example_idx]
for d in range(10):
    print(f"digit {d}: {example_errors[d]:.2f}")

print("predicted digit:", y_pred20[example_idx])


## Appendix: extra exploratory cells

The cells below are still related to the project, but they are not essential to the main classifier narrative above.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for d in range(10):
    ax.plot(np.arange(1, len(S[d]) + 1), S[d], label=f"digit {d}", linewidth=2)

ax.set_xlabel("Singular value index")
ax.set_ylabel("Singular value")
ax.set_title("Singular values of the digit matrices")
ax.set_xlim(1, 50)
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
S_avg = np.mean(np.array([S[d] for d in range(10)]), axis=0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.arange(1, len(S_avg) + 1), S_avg, linewidth=2)

ax.set_xlabel("Singular value index")
ax.set_ylabel("Average singular value")
ax.set_title("Average singular value decay across digits")
ax.set_xlim(1, 50)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
vectors = [(2, 2), (0.1, 2)]
titles = [r"Vector $(2,2)$", r"Vector $(0.1,2)$"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, (x, y), title in zip(axes, vectors, titles):
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.arrow(0, 0, x, y, head_width=0.08, head_length=0.12,
             length_includes_head=True, color="navy", linewidth=2)
    ax.arrow(0, 0, x, 0, head_width=0.06, head_length=0.10,
             length_includes_head=True, color="crimson", linewidth=2)
    ax.arrow(0, 0, 0, y, head_width=0.06, head_length=0.10,
             length_includes_head=True, color="forestgreen", linewidth=2)
    ax.plot([x, x], [0, y], linestyle="--", color="gray")
    ax.plot([0, x], [y, y], linestyle="--", color="gray")
    ax.text(x + 0.08, y + 0.08, f"({x},{y})", color="navy")
    ax.text(x / 2, -0.2, f"{x}", color="crimson", ha="center")
    ax.text(-0.2, y / 2, f"{y}", color="forestgreen", va="center")
    ax.set_title(title)
    ax.set_xlim(-0.5, 2.6)
    ax.set_ylim(-0.5, 2.6)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

fig.suptitle("A vector and its shadows on the coordinate axes")
plt.tight_layout()
plt.show()
